# SentinelVision AI
## Notebook 05B — Supervised Classifier on VideoMAE Embeddings

### Objective

Notebook 03 created VideoMAE embeddings from real surveillance video clips.

Notebook 04 tested unsupervised anomaly detection models on those embeddings.

This notebook tests whether the VideoMAE embeddings can support supervised binary classification:

`normal clip` vs `anomalous-source clip`

The model is not fine-tuning VideoMAE itself.

Instead, VideoMAE stays frozen and we train classifiers on top of the extracted embeddings.

In [1]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

Load Embeddings

In [2]:
embeddings_path = Path("../data/embeddings/clip_embeddings_videomae_sample.parquet")

embeddings = pd.read_parquet(embeddings_path)

print(embeddings.shape)
embeddings.head()

(600, 775)


,clip_id,video_id,category,binary_label,split,start_seconds,end_seconds,feature_000,feature_001,feature_002,...,feature_758,feature_759,feature_760,feature_761,feature_762,feature_763,feature_764,feature_765,feature_766,feature_767
0,Normal_Videos_365_x264_clip_00003,Normal_Videos_365_x264,Normal,0,test,12.0,16.0,0.178769,3.729066,3.666701,...,2.413040,-1.240642,-1.159237,-1.608377,3.872948,1.565241,0.879731,0.846557,-1.191670,-0.930276
1,Normal_Videos_365_x264_clip_00030,Normal_Videos_365_x264,Normal,0,test,120.0,124.0,0.278189,2.779260,3.542873,...,-0.741295,-1.844295,0.887464,0.550726,3.069808,1.138670,1.094854,0.417987,-1.605511,1.888869
2,Normal_Videos_365_x264_clip_00032,Normal_Videos_365_x264,Normal,0,test,128.0,132.0,0.229807,3.365491,3.970525,...,1.338005,-2.018188,-0.082708,-0.061900,3.764704,1.424313,0.843749,1.016206,-1.329999,1.080641
3,Normal_Videos_641_x264_clip_00015,Normal_Videos_641_x264,Normal,0,test,60.0,64.0,-5.002384,1.898196,0.756616,...,-2.385197,0.084327,4.650817,3.130729,2.914249,-0.534147,2.499364,3.860784,0.763254,-1.158123
4,Normal_Videos_312_x264_clip_00000,Normal_Videos_312_x264,Normal,0,test,0.0,4.0,-4.661326,1.897303,1.866576,...,0.865088,-1.091096,1.328566,4.970067,2.660053,1.487340,2.988191,1.090254,0.692304,0.847159


#### Verify balance

In [3]:
embeddings.groupby(["split", "binary_label"]).size()

split       binary_label
test        0               100
            1               100
train       0               100
            1               100
validation  0               100
            1               100
dtype: int64